In [ ]:
%pip install --quiet pandas pyarrow geopandas folium branca matplotlib

# Visualização — mapa interativo do output final

Mapa folium do parquet final `renda_setor_cep_sp_final.parquet`. Estilo igual ao **mapa final híbrido (Etapa 9)** do `pipeline_renda_geofencing_setor_cep.ipynb`:

- **Cor de preenchimento** = renda média estimada (`renda_v06004_estimada`); cinza = sem renda.
- **Borda**:
  - Preta fina → `origem_cep = geofencing` (caso saudável)
  - Azul média → `origem_cep = cnefe_original` (fallback para micro-setores em borda)
  - Vermelha grossa → `origem_cep = sem_endereco_cnefe` ou `tem_cep = 0` (raros)
- **Hover** mostra: Setor, Município, Distrito, Bairro, CD_TIPO, Renda média, Renda mediana, Origem renda, Origem CEP, Qtd CEPs, Faixa CEP, Endereços, primeiros 5 CEPs.

**Recorte**: por padrão usa o município com mais setores no output (SP-capital). Para mudar, ajuste `MAP_COD_MUNICIPIO`.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
from IPython.display import display

pd.set_option('display.max_columns', 120)


def find_project_root(start):
    start = Path(start).resolve()
    for c in [start, *start.parents]:
        if (c / 'saida_etl_final_sp').exists() and (c / 'BR_setores_CD2022').exists():
            return c
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
OUT_PARQUET  = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_final.parquet'
SHAPEFILE    = PROJECT_ROOT / 'BR_setores_CD2022' / 'BR_setores_CD2022.shp'

for p in [OUT_PARQUET, SHAPEFILE]:
    print(f'{p.name}: {"OK" if p.exists() else "NAO ENCONTRADO"}')

## Configuração do recorte

Mude `MAP_COD_MUNICIPIO` pra escolher outro município. `None` = pega o com mais setores.

In [ ]:
MAP_COD_MUNICIPIO = '3550308'  # SP capital. Alternativas: '3509502' Campinas, '3518800' Guarulhos, etc.
USAR_RENDA_ESTIMADA = True       # True = colorir por renda imputada+original. False = so renda original.
MAP_TOP_LABELS = 0               # Setores rotulados (0 = nenhum). Aumenta peso visual.

print({
    'MAP_COD_MUNICIPIO': MAP_COD_MUNICIPIO,
    'USAR_RENDA_ESTIMADA': USAR_RENDA_ESTIMADA,
    'MAP_TOP_LABELS': MAP_TOP_LABELS,
})

## Etapa 1 — Carregar parquet + geometria do recorte

In [ ]:
df = pd.read_parquet(OUT_PARQUET)
print(f'Linhas no parquet: {len(df):,}')

if MAP_COD_MUNICIPIO is None:
    MAP_COD_MUNICIPIO = (df.groupby('cod_mun').size().sort_values(ascending=False).index[0])
    print(f'Municipio escolhido auto: {MAP_COD_MUNICIPIO}')

recorte = df.loc[df['cod_mun'] == MAP_COD_MUNICIPIO].copy()
print(f'Setores no recorte: {len(recorte):,}')
print(f'  com renda original ou imputada: {int(recorte["renda_v06004_estimada"].notna().sum()):,}')
print(f'  sem renda: {int(recorte["renda_v06004_estimada"].isna().sum()):,}')
print(f'  com CEP: {int((recorte["tem_cep"] == 1).sum()):,}')

# Geometria do recorte.
print()
print(f'Carregando geometria do municipio {MAP_COD_MUNICIPIO}...')
geo = gpd.read_file(SHAPEFILE, where=f"CD_MUN = '{MAP_COD_MUNICIPIO}'")
geo['cd_setor'] = geo['CD_SETOR'].astype(str).str.strip()
print(f'Geometrias: {len(geo):,}')

# Merge LEFT do shapefile com o output. Cobre setores legitimamente sem renda tambem.
g = geo[['cd_setor', 'geometry']].merge(recorte, on='cd_setor', how='left')
if g.crs is not None and g.crs.to_epsg() != 4326:
    g = g.to_crs(epsg=4326)
print(f'Linhas no merge: {len(g):,}')

## Etapa 2 — Preparar dados para o mapa (cores e tooltip)

In [ ]:
# Encurta lista_ceps para tooltip.
def short_ceps(s):
    if s is None or (isinstance(s, float) and pd.isna(s)) or not s:
        return ''
    ceps = str(s).split('|')
    if len(ceps) <= 5:
        return ', '.join(ceps)
    return ', '.join(ceps[:5]) + f'  ... (+{len(ceps)-5})'


def fmt_brl(v):
    if v is None or pd.isna(v):
        return '-'
    s = f'{float(v):,.2f}'
    return 'R$ ' + s.replace(',', 'X').replace('.', ',').replace('X', '.')


def fmt_int(v):
    if v is None or pd.isna(v):
        return '-'
    return f'{int(v):,}'.replace(',', '.')


# Coluna usada para cor.
color_col = 'renda_v06004_estimada' if USAR_RENDA_ESTIMADA else 'renda_v06004'

# Sanitiza NaN nas colunas que vao pro tooltip.
g['Setor']          = g['cd_setor'].astype(str)
g['Municipio']      = g['nm_mun'].fillna('').astype(str)
g['Distrito']       = g['NM_DIST'].fillna('').astype(str)
g['Bairro']         = g['NM_BAIRRO'].fillna('').astype(str)
g['CD_TIPO_lbl']    = g['CD_TIPO'].fillna('').astype(str)
g['Situacao']       = g['SITUACAO'].fillna('').astype(str)
g['Renda media']    = g['renda_v06004_estimada'].apply(fmt_brl)
g['Renda mediana']  = g['renda_v06006_estimada'].apply(fmt_brl)
g['Origem renda']   = g['origem_renda'].fillna('').astype(str)
g['Origem CEP']     = g['origem_cep'].fillna('').astype(str)
g['Qtd CEPs']       = g['qtd_ceps'].apply(fmt_int)
g['Faixa CEP']      = g['faixa_cep'].fillna('').astype(str)
g['Enderecos']      = g['total_enderecos'].apply(fmt_int)
g['CEPs']           = g['lista_ceps'].apply(short_ceps)

# Snapshot das colunas numericas ANTES de virarem object.
# Crucial: preservar None nessas colunas para o style_function distinguir "sem renda".
num_cols = list(g.select_dtypes(include=['number']).columns)

# NaN em colunas numericas -> None pra geojson (folium serializa None como null).
for col in num_cols:
    g[col] = g[col].astype(object).where(g[col].notna(), None)

# Strings vazias apenas em colunas TEXTUAIS (exclui as que acabaram de virar object).
str_cols = [
    c for c in g.columns
    if c not in num_cols
    and c != 'geometry'
    and pd.api.types.is_object_dtype(g[c])
]
for col in str_cols:
    g[col] = g[col].fillna('')

print(f'Setores prontos pro mapa: {len(g):,}')
print()
print('Distribuicao origem_renda no recorte:')
if 'origem_renda' in recorte.columns:
    print(recorte['origem_renda'].value_counts())


## Etapa 3 — Mapa interativo folium

In [ ]:
# Colormap pela renda (apenas valores nao-nulos).
renda_vals = pd.to_numeric(
    pd.Series([v for v in g[color_col] if v is not None]),
    errors='coerce',
).dropna()
if len(renda_vals):
    colormap = cm.linear.YlOrRd_09.scale(float(renda_vals.min()), float(renda_vals.max()))
    colormap.caption = f'{color_col} (R$)'
else:
    colormap = None

# Estilo por origem_cep (borda).
ORIGEM_STYLE = {
    'geofencing':         {'color': 'black', 'weight': 0.4},
    'cnefe_original':     {'color': 'blue',  'weight': 1.2},
    'sem_endereco_cnefe': {'color': 'red',   'weight': 2.5},
}
DEFAULT_BORDER = {'color': 'gray', 'weight': 1.0}


def style_function(feature):
    p = feature['properties']
    r = p.get(color_col)
    origem = p.get('origem_cep') or ''
    fill = '#cccccc' if r is None else (colormap(r) if colormap else '#999999')
    s = ORIGEM_STYLE.get(origem, DEFAULT_BORDER)
    return {
        'fillColor': fill,
        'color':     s['color'],
        'weight':    s['weight'],
        'fillOpacity': 0.75,
    }


center = g.geometry.union_all().centroid
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=12,
    tiles='cartodbpositron',
    control_scale=True,
)

tooltip_pairs = [
    ('Setor',         'Setor:'),
    ('Municipio',     'Municipio:'),
    ('Distrito',      'Distrito:'),
    ('Bairro',        'Bairro:'),
    ('Situacao',      'Situacao:'),
    ('CD_TIPO_lbl',   'CD_TIPO:'),
    ('Renda media',   'Renda media:'),
    ('Renda mediana', 'Renda mediana:'),
    ('Origem renda',  'Origem renda:'),
    ('Origem CEP',    'Origem CEP:'),
    ('Qtd CEPs',      'Qtd CEPs:'),
    ('Faixa CEP',     'Faixa CEP:'),
    ('Enderecos',     'Enderecos:'),
    ('CEPs',          'CEPs:'),
]
fields  = [f for f, _ in tooltip_pairs if f in g.columns]
aliases = [a for f, a in tooltip_pairs if f in g.columns]

folium.GeoJson(
    g,
    name='setores',
    style_function=style_function,
    tooltip=folium.features.GeoJsonTooltip(
        fields=fields, aliases=aliases, sticky=True, max_width=420,
    ),
).add_to(m)

if colormap is not None:
    colormap.add_to(m)

# Legenda de bordas.
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid #888;
            font-family: sans-serif; font-size: 12px; line-height: 18px;">
  <b>Origem do CEP</b><br>
  <span style="display:inline-block; width:18px; height:3px; background:black; vertical-align:middle;"></span> geofencing<br>
  <span style="display:inline-block; width:18px; height:3px; background:blue;  vertical-align:middle;"></span> cnefe_original<br>
  <span style="display:inline-block; width:18px; height:3px; background:red;   vertical-align:middle;"></span> sem_endereco_cnefe<br>
  <hr style="margin: 4px 0;">
  <b>Preenchimento</b><br>
  cinza = sem renda (legitimo ou nao imputavel)<br>
  gradiente = renda media (V06004)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Salva o mapa em HTML standalone (abre fora do notebook, sem depender de output do Jupyter).
MAP_OUT_DIR = PROJECT_ROOT / 'saida_etl_final_sp'
MAP_OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_HTML = MAP_OUT_DIR / f'mapa_renda_{MAP_COD_MUNICIPIO}.html'
m.save(str(MAP_HTML))

print(f'Mapa montado: {len(g):,} setores no municipio {MAP_COD_MUNICIPIO}')
print(f'HTML salvo em: {MAP_HTML}')
m


## Notas

- **Trocar município**: ajuste `MAP_COD_MUNICIPIO` na célula de config e re-execute as 3 células seguintes.
- **Visualizar só renda original** (sem imputação): troque `USAR_RENDA_ESTIMADA = False`.
- **Performance**: 27k polígonos em SP-capital renderizam em ~5-8 segundos no navegador. Para municípios pequenos é instantâneo.
- **Setores cinza**: são os 2,75% legítimos sem renda (`origem_renda` começa com `sem_dado_legitimo_`). Ver `2_analises_diagnostico.ipynb` para detalhes.